# Mi primer bot de telegram con python

## import de librerias

In [1]:
import telebot
from dotenv import load_dotenv
import os
import time
import random
import json
from telebot import types
from telebot.types import InlineKeyboardMarkup
from telebot.types import InlineKeyboardButton
from telebot.types import ForceReply
from comandos.common import ErrorUsuario
from comandos.estrategia import responder_estrategia, manejar_callback
from comandos.quepiloto import responder_quepiloto
import comandos.estrategia as e
import comandos.historico as h
import comandos.quepiloto as qp

## Instanciar el bot

In [2]:
load_dotenv()
tkn= os.getenv("TELEGRAM_TOKEN")
bot_inicio= telebot.TeleBot(tkn)

## Funciones de comandos
hay dos tipos de comandos: 
                   -los que empiezan con "/"
                   -cada comando tipo de mensaje :{texto,imagen,photo(cuando la sacás con la cámara), voice}


In [3]:
@bot_inicio.message_handler(commands=["text"])
def responder_html(mensaje):
    #Primera rta a enviar: html
        rta_html=""
    #Con etiquetas concatenadas:
        rta_html+="<b><u>Título formato HTML </u>:</b>"+"\n"
    #Forma de generar una rta con formato html
        rta_html += "<b> Texto en negrita </b>" +"\n"
        rta_html += "<i> Texto en cursiva </i>" +"\n"
        rta_html += "<u> Texto subrayado </u>" +"\n"
        rta_html += "<s> Texto tachado </s>" +"\n"
        rta_html += "<code> Monoespaciado </code>" +"\n"
        rta_html += '<a href="http://www.formula1.com/"> Link a web </a>' +"\n"
        rta_html += "<span class= 'tg-spoiler'> Texto spoiler </span>"
@bot_inicio.message_handler(commands= ["text"])
def responder_mark(mensaje):    
    #Segunda respuesta a enviar: formato markdown
        rta_mark=""
    #Con etiquetas concatenadas:
        rta_mark += "*__Título formato Markdown__:*"+"\n"
        rta_mark += "_Texto cursiva_:"+"\n"
        rta_mark += "__Text subrayado__:"+"\n"
        rta_mark += "~Texto tachado~"+"\n"
        rta_mark += "```Monoespacio```"+"\n"
        rta_mark += '[link a web] ("http://www.formula1.com/")'+"\n"
        rta_mark += "||Texto spoiler||"

In [4]:

@bot_inicio.message_handler(commands=["start"])
def cmd_start(message):
    nombre = message.from_user.first_name
    bot_inicio.send_message(
        message.chat.id,
        f"¡Hola, {nombre}! 🏎️ Soy el bot de Fórmula 1.\n"
        "Puedo simular estrategias de neumáticos, contarte qué piloto nació en tu mes "
        "y mostrarte resultados históricos.\n\n"
        "Escribí /ayuda para ver todos los comandos 🏁"
    )

@bot_inicio.message_handler(commands=["saludar"])
def cmd_saludar(message):
    print("me enviaron /saludar")
    bot_inicio.reply_to(message,"Hola, soy un bot. Con que te gustaría comenzar?")



@bot_inicio.message_handler(commands=["ayuda"])
def ayuda(message):
     bot_inicio.reply_to(message,     "Soy un🏎️ Bot de F1\n\n"
                         "Que te puedo ayudar con:\n"
                         "/estrategia (Generemos una estrategia?) <año> <carrera> <piloto>\n   ej: /estrategia 2024 belgica HAM\n"
                         "/quepiloto (Con que piloto compartís el mes?) <mes>\n   ej: /quepiloto marzo\n\n"
                         "/historico (Aprendemos de historia?) <año> [carrera]\n   ej: /historico 2008  |  /historico 1994 brasil"
)

In [5]:
@bot_inicio.message_handler(commands="botones")
def cmd_botones(message):
    markup=InlineKeyboardMarkup(row_width=2)
    b1 = InlineKeyboardButton("Google", url="google.com.ar") 
    b2 = InlineKeyboardButton(" Página web - Fórmula 1", url="https://www.formula1.com/")
    b3 =InlineKeyboardButton("Instagram - Fórmula 1", url="https://www.instagram.com/f1/")
    b4=InlineKeyboardButton("Twitter - Fórmula 1", url="https://x.com/F1")
    markup.add(b1,b2,b3,b4)
    bot_inicio.send_message(message.chat.id, "Tus principales atajos son:", reply_markup=markup)

2026-10-31 20:29:53,106 (__init__.py:9465 MainThread) ERROR - TeleBot: "message_handler: Commands filter should be list of strings (commands), unknown type supplied to the 'commands' filter list. Not able to use the supplied type."


In [6]:
def crear_teclado(filas):
    teclado = InlineKeyboardMarkup()
    for fila in filas:
        botones = []
        for texto, dato in fila:
            boton = InlineKeyboardButton(
                texto,
                callback_data=dato
            )
            botones.append(boton)
        teclado.row(*botones)
    return teclado

FILE_ID_INTRO = "BAACAgEAAxkBAAO4auZW0WevJ1rQWetfLMXFgtAwrY4AAvAJAAJNHTFH-qZwp_9t53E9BA"

@bot_inicio.message_handler(commands=["estrategia"])
def estrategia(message):
    args = message.text.split()[1:]
    try:
        texto, filas = responder_estrategia(args)
        teclado = crear_teclado(filas) if filas else None

        try:
            bot_inicio.send_video(message.chat.id, FILE_ID_INTRO)
        except Exception as e:
            print("No se pudo enviar el video:", e)

        bot_inicio.send_message(message.chat.id, texto, reply_markup=teclado)
    except ErrorUsuario as error:
        bot_inicio.send_message(message.chat.id, str(error))

@bot_inicio.callback_query_handler(
    func=lambda call: call.data.startswith("est:")
)
def responder_boton(call):
    try:
        texto, filas = manejar_callback(call.data)
        if filas:
            teclado = crear_teclado(filas)
        else:
            teclado = None
        bot_inicio.edit_message_text(
            texto,
            call.message.chat.id,
            call.message.message_id,
            reply_markup=teclado
        )
        bot_inicio.answer_callback_query(call.id)
    except ErrorUsuario as error:
        bot_inicio.answer_callback_query(
            call.id,
            str(error),
            show_alert=True
        )

In [7]:
def _pedir_mes(chat_id, texto=qp.PREGUNTA_MES):
    msg = bot_inicio.send_message(chat_id, texto, reply_markup=ForceReply())
    bot_inicio.register_next_step_handler(msg, _recibir_mes)


def _responder_mes(chat_id, texto):
    bot_inicio.send_chat_action(chat_id, "typing")
    try:
        bot_inicio.send_message(chat_id, qp.responder_quepiloto(texto))
    except ErrorUsuario as error:
        bot_inicio.send_message(chat_id, str(error))
    except Exception as error:
        print("Error en /quepiloto:", error)
        bot_inicio.send_message(chat_id, "No pude traer los datos ahora. Probá de nuevo en un rato.")


def _recibir_mes(message):
    if message.content_type != "text":
        return _pedir_mes(message.chat.id, "Necesito el mes escrito, por ejemplo: marzo o 3.")
    if message.text.startswith("/"):
        bot_inicio.process_new_messages([message])   # era otro comando: que lo atienda su handler
        return
    if qp.interpretar_mes(message.text) is None:
        return _pedir_mes(message.chat.id, "No reconocí ese mes 😅 Probá con el nombre (marzo) o el número (3).")
    _responder_mes(message.chat.id, message.text)


@bot_inicio.message_handler(commands=["quepiloto"])
def quepiloto(message):
    partes = message.text.split(maxsplit=1)
    if len(partes) > 1:                              # /quepiloto marzo (modo directo)
        _responder_mes(message.chat.id, partes[1])
    else:
        _pedir_mes(message.chat.id)

qp.pilotos_por_mes()

{3: [{'driverId': 'alguersuari',
   'code': 'ALG',
   'url': 'http://en.wikipedia.org/wiki/Jaime_Alguersuari',
   'givenName': 'Jaime',
   'familyName': 'Alguersuari',
   'dateOfBirth': '1990-03-23',
   'nationality': 'Spanish'},
  {'driverId': 'glock',
   'code': 'GLO',
   'url': 'http://en.wikipedia.org/wiki/Timo_Glock',
   'givenName': 'Timo',
   'familyName': 'Glock',
   'dateOfBirth': '1982-03-18',
   'nationality': 'German'},
  {'driverId': 'maldonado',
   'permanentNumber': '13',
   'code': 'MAL',
   'url': 'http://en.wikipedia.org/wiki/Pastor_Maldonado',
   'givenName': 'Pastor',
   'familyName': 'Maldonado',
   'dateOfBirth': '1985-03-09',
   'nationality': 'Venezuelan'},
  {'driverId': 'merhi',
   'permanentNumber': '98',
   'code': 'MER',
   'url': 'http://en.wikipedia.org/wiki/Roberto_Merhi',
   'givenName': 'Roberto',
   'familyName': 'Merhi',
   'dateOfBirth': '1991-03-22',
   'nationality': 'Spanish'},
  {'driverId': 'vandoorne',
   'permanentNumber': '2',
   'code': 'VA

In [8]:
@bot_inicio.message_handler(commands=["historico"])
def historico(message):
    args = message.text.split()[1:]
    try:
        texto, filas = h.responder_historico(args)
        teclado = crear_teclado(filas) if filas else None
        bot_inicio.send_message(message.chat.id, texto, reply_markup=teclado)
    except ErrorUsuario as error:
        bot_inicio.send_message(message.chat.id, str(error))
    except Exception as error:
        print("Error en /historico:", error)
        bot_inicio.send_message(message.chat.id, "No pude traer los datos ahora. Probá de nuevo en un rato.")


@bot_inicio.callback_query_handler(func=lambda call: call.data.startswith("his:"))
def responder_boton_historico(call):
    try:
        texto, filas = h.manejar_callback(call.data)
        teclado = crear_teclado(filas) if filas else None
        bot_inicio.edit_message_text(
            texto,
            call.message.chat.id,
            call.message.message_id,
            reply_markup=teclado
        )
        bot_inicio.answer_callback_query(call.id)
    except ErrorUsuario as error:
        bot_inicio.answer_callback_query(call.id, str(error), show_alert=True)
    except Exception as error:
        print("Error en callback de historico:", error)
        bot_inicio.answer_callback_query(
            call.id,
            "No pude traer los datos ahora. Probá de nuevo en un rato.",
            show_alert=True
        )

In [9]:
@bot_inicio.message_handler(commands=["dolar"])
def cmd_dolar(message):
    bot_inicio.send_chat_action(message.chat.id, "typing")
    try:
        r = requests.get("https://dolarapi.com/v1/dolares", timeout=10)
        r.raise_for_status()
        cotizaciones = r.json()
    except requests.exceptions.RequestException:
        bot_inicio.reply_to(message, "😕 No pude consultar la cotización ahora. Probá en un rato.")
        return

    texto = "💵 <b>Cotización del dólar</b>\n\n"
    for c in cotizaciones:
        compra = c.get("compra")
        venta = c.get("venta")
        compra_txt = f"${compra:,.2f}" if compra else "-"
        venta_txt = f"${venta:,.2f}" if venta else "-"
        texto += f"<b>{c['nombre']}</b>: compra {compra_txt} | venta {venta_txt}\n"

    fecha = cotizaciones[0].get("fechaActualizacion", "")[:16].replace("T", " ")
    texto += f"\n<i>Actualizado: {fecha}</i>"
    bot_inicio.send_message(message.chat.id, texto, parse_mode="HTML")

In [10]:
@bot_inicio.message_handler(content_types=["voice", "audio"])
def responder_audio(message):
    if message.content_type == "voice":
        duracion = message.voice.duration
        tipo = "nota de voz"
    else:
        duracion = message.audio.duration
        tipo = "audio"
    bot_inicio.reply_to(
        message,
        f"🎧 Recibí tu {tipo} de {duracion} segundos.\n"
        "Por ahora no puedo escuchar audios, pero escribime /ayuda y te muestro lo que sé hacer 🏎️"
    )

In [11]:
@bot_inicio.message_handler(content_types=["document"])
def responder_documento(message):
    doc = message.document
    tamanio_kb = doc.file_size / 1024 if doc.file_size else 0
    bot_inicio.reply_to(
        message,
        f"📄 Recibí el archivo <b>{doc.file_name}</b> ({tamanio_kb:.1f} KB).\n"
        "Gracias, pero solo sé hablar de Fórmula 1 🏁 Probá con /ayuda",
        parse_mode="HTML"
    )

In [12]:
@bot_inicio.message_handler(commands=["fin"])
def cmd_fin(message):
    bot_inicio.reply_to(message,"Hasta la proxima")

@bot_inicio.message_handler(commands=["problema"])
def cmd_problema(message):
    print("alguien tiene un problema")
    bot_inicio.reply_to(message,"Ok, ya se lo pase a un admin, en un rato será solucionado")
    print("se envio el problema del usuario")
    print(message)

    usuario = message.from_user.username
    nombre= message.from_user.first_name

    if usuario:
        identificador = f"@{usuario}"
    else:
        identificador = nombre

    msg = f"El usuario: {identificador}, (id: {message.chat.id}) tiene un problema"
    bot_inicio.send_message(8656118293,msg)
    bot_inicio.send_message(8797897882,msg)
    bot_inicio.send_message( 8695615052,msg)
@bot_inicio.message_handler(content_types=["text"])
def responder_mensaje(message):
    print("me enviaron un texto")
    if message.text.startswith("/"):
        bot_inicio.send_message(message.chat.id, "Comando inexistente")
    else:
         bot_inicio.send_message(message.chat.id, "me enviaste un texto, que todavia no puedo responder")


## Setear comandos
Acá se retiten los de acceso directo, de los 50 que tenés, se muestran 5 que son los más utilizados

In [13]:
bot_inicio.set_my_commands([
    telebot.types.BotCommand("saludar", "Saludo del bot"),
    telebot.types.BotCommand("estrategia", "Generá una estrategia de carrera"),
    telebot.types.BotCommand("quepiloto", "Descubrí con qué piloto compartís el mes"),
    telebot.types.BotCommand("historico", "Repasá la historia de la F1 por año"),
    telebot.types.BotCommand("historial", "Mirá tu historial de consultas"),
    telebot.types.BotCommand("problema", "Pedí ayuda a un admin"),
    telebot.types.BotCommand("botones", "Atajos a las webs de la F1"),
    telebot.types.BotCommand("fin", "Finalizar")
])

True

## Lanzar el bot
Lanzamos el bot utilizando el while true
Y cuando estemos trabajando con el bot API, vamos a usar, infinity polling

In [14]:
if __name__== "__main__":
    print("Iniciando el bot")
    bot_inicio.infinity_polling()
    print("Fin")

Iniciando el bot


2026-10-31 20:39:56,380 (__init__.py:1160 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2026-10-31 20:39:56,382 (__init__.py:1162 MainThread) ERROR - TeleBot: "Break infinity polling"


Fin
